<a href="https://colab.research.google.com/github/bsheese/cs377/blob/main/17_regression_crossval/17_2_MLR/17_2_4_1_MLR_Ames_Part1_Revised.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MLR Predicting Housing Prices in Ames Iowa: Part 1,  Data Cleaning

Author: Brad Sheese
___

## Purpose of this Notebook
This notebook demonstrates manual cleaning of a real-world dataset. The intent of the cleaning is to prepare the dataset for inference, wherein we are interested in examining individual coefficients of features in the model.

### Data Cleaning, Under and Over Cleaning
The primary goal of data cleaning is to transform messy, raw data into a structured, model-ready format that maximizes true predictive signals and minimizes disruptive noise. This involves transforming the data in different ways. The trick is that we need to do this  without distorting the underlying reality of the information.

Achieving this balance requires navigating the dual pitfalls of under-cleaning and over-cleaning. Under-cleaning leaves too much chaos in the dataset, such as unhandled missing values, unencoded strings, extreme outliers, or highly collinear features, which can cause machine learning algorithms to crash, fail to converge, or overfit on structural errors. Conversely, over-cleaning stems from an overzealous attempt to simplify or shrink the data, resulting in the destruction of valuable information through the aggressive dropping of features, the clumsy grouping of highly distinct categories into generic buckets, or inappropriate binarization of continuous variables.

### Cleaning Specifically with MLR in Mind
Preparing data specifically for multiple linear regression (MLR)  requires strict adherence to the algorithm’s statistical assumptions.
* Because MLR relies on ordinary least squares (OLS) optimization, it is highly sensitive to missing values and extreme outliers, both of which can disproportionately pull the line of best fit and must be carefully imputed, capped, or removed.
* Since the MLR equation requires strictly numeric inputs, categorical data must be converted into binary flags using one-hot encoding, with the critical step of dropping one baseline category to avoid the "dummy variable trap" (perfect multicollinearity).
* Mitigating multicollinearity is a primary focus of MLR
cleaning. We routinely use Variance Inflation Factor (VIF) scores to
identify and remove highly correlated overlapping predictors that would
otherwise cause the model's coefficient estimates to become wildly unstable and
meaningless.
* Applying mathematical transformations (such as logarithmic
scaling) to highly skewed variables helps satisfy the strict assumptions of
linearity and homoscedasticity, ensuring the final model is not just
mathematically functional, but statistically valid and interpretable.

---

## Data for this Exercise
**Data Source:** http://jse.amstat.org/v19n3/decock/AmesHousing.txt

The Ames housing dataset, compiled by Dean De Cock in 2011, is a comprehensive collection of information detailing nearly every aspect of residential property sales in Ames, Iowa, from 2006 to 2010. It consists of approximately 2,930 observations and 79 explanatory variables, ranging from house quality and neighborhood to square footage and number of bathrooms, which are used to predict the final "SalePrice" of each home.

## Phase 1: Structural Fixes & Semantic Cleaning (Pre-Split)
*These steps are deterministic (they don't rely on dataset statistics) and can safely be done on the whole dataset before splitting.*

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

# Loading the dataframe
url = 'https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_housing_ames.txt'
df = pd.read_csv(url, sep='\t')

df.head()

In [ ]:
# 1. Drop Identifiers
df = df.drop(['Order', 'PID'], axis=1)

# Drop near-constant or highly sparse features to keep the dataset manageable for beginners
sparse_drops = ['Pool QC', 'Pool Area', 'Misc Feature', 'Misc Val', 'Alley', 'Fence']
constant_drops = ['Street', 'Utilities', 'Condition 2', 'Roof Matl', 'Heating', 'Low Qual Fin SF', '3Ssn Porch']
# We will also drop 'Garage Yr Blt' to simplify and avoid leakage
df = df.drop(sparse_drops + constant_drops + ['Garage Yr Blt'], axis=1, errors='ignore')

# 2. Correct Data Types
# MSSubClass is a numeric code, not a magnitude. Convert to string so it's not treated as a continuous number.
df['MS SubClass'] = df['MS SubClass'].astype(str)

# 3. Resolve "Meaningful" NAs
# Basement NAs mean "No Basement"
bsmt_cols = ['Bsmt Qual', 'Bsmt Cond', 'Bsmt Exposure', 'BsmtFin Type 1', 'BsmtFin Type 2']
for col in bsmt_cols:
    df[col] = df[col].fillna('None')

# Garage NAs mean "No Garage"
garage_cols = ['Garage Type', 'Garage Finish', 'Garage Qual', 'Garage Cond']
for col in garage_cols:
    df[col] = df[col].fillna('None')

# Fireplace NAs mean "No Fireplace"
df['Fireplace Qu'] = df['Fireplace Qu'].fillna('None')

df.info()

## Phase 2: The Golden Rule (Train/Test Split)
Before calculating any statistics, computing averages, or scaling data, we must split our data into training and test sets. Everything from this point forward must be learned *only* from the training set to prevent future data from leaking into your model.

In [ ]:
# 4. Train/Test Split
df_train, df_test = train_test_split(df, test_size=0.2, random_state=42)

print(f"Training set: {df_train.shape}")
print(f"Test set: {df_test.shape}")

## Phase 3: Handling Missing and Extreme Values (Train-Set Only)
Now we handle extreme outliers and true missing values. The dataset creator explicitly recommends removing houses with over 4,000 sq ft living area, as they represent unusual sales that skew OLS regression.

In [ ]:
# 5. Remove Extreme Outliers (Train set only)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.scatterplot(x=df_train['Gr Liv Area'], y=df_train['SalePrice'], ax=ax[0])
ax[0].set_title("Before Outlier Removal")

df_train = df_train[df_train['Gr Liv Area'] <= 4000].copy()

sns.scatterplot(x=df_train['Gr Liv Area'], y=df_train['SalePrice'], ax=ax[1])
ax[1].set_title("After Outlier Removal")
plt.show()

In [ ]:
# 6. Statistical Imputation
# Calculate medians on the training set ONLY
numeric_cols = df_train.select_dtypes(include=np.number).columns
cols_with_na = numeric_cols[df_train[numeric_cols].isna().any()].tolist()

print(f"Imputing missing values for: {cols_with_na}")

for col in cols_with_na:
    median_val = df_train[col].median()
    df_train[col] = df_train[col].fillna(median_val)
    df_test[col] = df_test[col].fillna(median_val) # Apply train median to test set

## Phase 4: Feature Engineering & Transformations
MLR assumes a linear relationship. We can use log transformations on heavily skewed variables like SalePrice. We can also combine related features to reduce dimensionality.

In [ ]:
# 7. Transform Skewed Variables
df_train['Log_SalePrice'] = np.log(df_train['SalePrice'])
df_test['Log_SalePrice'] = np.log(df_test['SalePrice'])

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df_train['SalePrice'], ax=ax[0], kde=True)
ax[0].set_title('Original SalePrice (Right-Skewed)')
sns.histplot(df_train['Log_SalePrice'], ax=ax[1], kde=True)
ax[1].set_title('Log-Transformed SalePrice (Normal-ish)')
plt.show()

# Drop the original SalePrice so the model doesn't use it
df_train = df_train.drop('SalePrice', axis=1)
df_test = df_test.drop('SalePrice', axis=1)

In [ ]:
# 8. Create Synergistic Features
df_train['Total_Square_Footage'] = df_train['1st Flr SF'] + df_train['2nd Flr SF'] + df_train['Total Bsmt SF']
df_test['Total_Square_Footage'] = df_test['1st Flr SF'] + df_test['2nd Flr SF'] + df_test['Total Bsmt SF']

# Drop the components to avoid multicollinearity
df_train = df_train.drop(['1st Flr SF', '2nd Flr SF', 'Total Bsmt SF'], axis=1)
df_test = df_test.drop(['1st Flr SF', '2nd Flr SF', 'Total Bsmt SF'], axis=1)

## Phase 5: Encoding for the Algorithm
Because MLR requires strict numeric inputs, we must translate text to math. We use Ordinal Encoding for ranked variables, and One-Hot Encoding (Nominal) for unranked variables.

In [ ]:
# 9. Ordinal Encoding
quality_map = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0}
ordinal_cols = ['Exter Qual', 'Kitchen Qual', 'Heating QC', 'Bsmt Qual', 'Fireplace Qu']

for col in ordinal_cols:
    df_train[col] = df_train[col].map(quality_map).fillna(3) # TA is typical/average
    df_test[col] = df_test[col].map(quality_map).fillna(3)

# Garage Finish has a different mapping
garage_map = {'Fin': 3, 'RFn': 2, 'Unf': 1, 'None': 0}
df_train['Garage Finish'] = df_train['Garage Finish'].map(garage_map).fillna(0)
df_test['Garage Finish'] = df_test['Garage Finish'].map(garage_map).fillna(0)

In [ ]:
# 10. Nominal Encoding (One-Hot)
# CRUCIAL: use drop_first=True to avoid the "Dummy Variable Trap" (perfect multicollinearity)
nominal_cols = ['Neighborhood', 'Foundation', 'MS Zoning', 'Bldg Type', 'Central Air']

# To ensure columns match exactly between train and test, it's safer to concat, encode, then split back.
df_combined = pd.concat([df_train, df_test])
df_combined = pd.get_dummies(df_combined, columns=nominal_cols, drop_first=True)

# Drop any remaining unencoded object columns for simplicity in our MLR model
objects_to_drop = df_combined.select_dtypes(include='object').columns
df_combined = df_combined.drop(objects_to_drop, axis=1)

df_train = df_combined.loc[df_train.index]
df_test = df_combined.loc[df_test.index]

## Phase 6: Multicollinearity Check
If two independent variables are highly correlated (e.g., GarageCars and GarageArea), they confuse the model. We can run a Variance Inflation Factor (VIF) check or examine correlation directly.

In [ ]:
# 11. Multicollinearity Check
print("Correlation between Garage Cars and Garage Area:",
      df_train['Garage Cars'].corr(df_train['Garage Area']))

corr_with_price = df_train[['Garage Cars', 'Garage Area', 'Log_SalePrice']].corr()['Log_SalePrice']
print("\nCorrelation with Log_SalePrice:")
print(corr_with_price)

# Because they are highly correlated (~0.89), we drop the one less correlated with Log_SalePrice (Garage Area)
df_train = df_train.drop('Garage Area', axis=1)
df_test = df_test.drop('Garage Area', axis=1)

## Summary
By following this order—**Structure → Split → Impute → Transform → Encode → Prune**—we ensured that our model remains statistically pure (no data leakage), retains its valuable signals (proper encoding), and respects the strict mathematical boundaries of Multiple Linear Regression (no multicollinearity or outliers).

### Where This Work Goes Next
All of the cleaning steps we walked through here have been packaged into a reusable module called `ames_cleaning.py`. In Part 2, we will load that module and call `load_and_clean_ames()`, which applies every decision we just made, so we can focus on building and evaluating models without re-running all of this cleaning code. The function exists to keep the modeling notebooks focused, but everything it does is exactly what we practiced here.